# Project 09 - Cell-Cell Communication Across Breast Cancer Subtypes (CellChat)

**Dataset:** Wu et al. 2021 breast cancer atlas, 3 patients spanning 3 molecular subtypes (CID3921 = ER+, CID4290A = HER2+, CID4515 = TNBC), 12,962 cells, 9 cell types.

**Method:** CellChat v2 (Jin et al. 2024) - the canonical R tool for cell-cell communication inference from scRNA-seq.

**Goal:** quantify ligand-receptor signaling between cell populations in the tumor microenvironment, then compare TNBC vs HER2+ vs ER+ to identify subtype-specific signaling rewiring. This is a functional analysis that builds on the cell-type annotations already validated in Project 02.

**Pipeline:**
1. Load Wu 2021 h5ad, split by subtype
2. Per-subtype CellChat: identifyOverExpressedGenes -> computeCommunProb -> aggregateNet
3. Merge into one multi-dataset CellChat object
4. Comparison: total interactions, differential interaction networks, pathway ranking
5. Hero figure + supporting visualizations + per-subtype top L-R pairs

## 1. Setup

In [ ]:
suppressPackageStartupMessages({
  library(CellChat)
  library(Seurat)
  library(SeuratDisk)
  library(anndata)
  library(tidyverse)
  library(patchwork)
  library(ComplexHeatmap)
  library(NMF)
  library(circlize)
  library(Matrix)
})

options(stringsAsFactors = FALSE)
set.seed(42)
options(future.globals.maxSize = 4 * 1024^3)  # 4 GiB for CellChat parallelism

FIG_DIR <- '../figures'
RES_DIR <- '../results'
dir.create(FIG_DIR, showWarnings = FALSE, recursive = TRUE)
dir.create(RES_DIR, showWarnings = FALSE, recursive = TRUE)

save_fig <- function(p, path, w = 8, h = 6, dpi = 300) {
  ggsave(path, plot = p, width = w, height = h, dpi = dpi, bg = 'white')
}

cat('CellChat:    ', as.character(packageVersion('CellChat')), '\n')
cat('Seurat:      ', as.character(packageVersion('Seurat')), '\n')
cat('SeuratDisk:  ', as.character(packageVersion('SeuratDisk')), '\n')

## 2. Load Wu 2021 data

We load the integrated h5ad from Project 02 via the `anndata` R package, then build the expression matrix and metadata needed by CellChat. CellChat needs log-normalized expression (counts/sum * 10^4, log-transformed) - the Wu 2021 .X is already log-normalized per the dataset spec.

In [ ]:
h5ad_path <- '/home/marko-b2/upwork_portfolio/02_breast_cancer_TME/results/wu2021_3patient_TME_processed.h5ad'
ad <- read_h5ad(h5ad_path)

cat('AnnData shape:', dim(ad), '\n')
cat('obs columns:\n')
print(colnames(ad$obs))
cat('\nSubtype distribution:\n')
print(table(ad$obs$subtype))
cat('\nCell type distribution:\n')
print(table(ad$obs$celltype_major))

In [ ]:
# Build cell x gene log-normalized matrix as dgCMatrix (cells x genes -> needs transpose for CellChat which expects genes x cells)
X <- ad$X
if (!inherits(X, 'dgCMatrix')) {
  X <- as(X, 'CsparseMatrix')
}
# anndata stores cells x genes; CellChat expects genes x cells
data.input <- Matrix::t(X)
rownames(data.input) <- ad$var_names
colnames(data.input) <- ad$obs_names

meta <- ad$obs
meta$cell_id <- rownames(meta)
meta$labels <- as.character(meta$celltype_major)
meta$subtype <- as.character(meta$subtype)

stopifnot(all(colnames(data.input) == meta$cell_id))

cat('Expression matrix:', dim(data.input)[1], 'genes x', dim(data.input)[2], 'cells\n')
cat('Max expression value:', round(max(data.input), 2), '(should be ~5-10 for log-norm)\n')

## 3. Subtype split

In [ ]:
subtypes <- c('ER+', 'HER2+', 'TNBC')

split_data <- list()
split_meta <- list()
for (s in subtypes) {
  cells <- meta$cell_id[meta$subtype == s]
  split_data[[s]] <- data.input[, cells, drop = FALSE]
  split_meta[[s]] <- meta[meta$subtype == s, ]
  cat(sprintf('%-6s %d cells, %d cell types\n', s, ncol(split_data[[s]]),
              length(unique(split_meta[[s]]$labels))))
}

## 4. Per-subtype CellChat pipeline

For each subtype, build a CellChat object, identify over-expressed signaling genes and L-R pairs, compute communication probability and aggregate the network. We use CellChatDB.human (~2000 ligand-receptor pairs, secreted signaling + ECM-receptor + cell-cell contact).

In [ ]:
CellChatDB <- CellChatDB.human
cat('CellChatDB.human:', nrow(CellChatDB$interaction), 'interactions\n')
cat('Interaction categories:\n')
print(table(CellChatDB$interaction$annotation))

In [ ]:
cellchat_list <- list()

for (s in subtypes) {
  cat('\n====', s, '====\n')
  cc <- createCellChat(object = split_data[[s]], meta = split_meta[[s]], group.by = 'labels')
  cc@DB <- CellChatDB
  
  # Subset to signaling genes only
  cc <- subsetData(cc)
  future::plan('multisession', workers = 4)
  
  cat('Identifying over-expressed signaling genes/interactions...\n')
  cc <- identifyOverExpressedGenes(cc)
  cc <- identifyOverExpressedInteractions(cc)
  
  cat('Computing communication probability (truncatedMean, this is the slow step)...\n')
  cc <- computeCommunProb(cc, type = 'truncatedMean', trim = 0.1, raw.use = TRUE)
  cc <- filterCommunication(cc, min.cells = 10)
  
  cat('Computing pathway-level probability and aggregating network...\n')
  cc <- computeCommunProbPathway(cc)
  cc <- aggregateNet(cc)
  cc <- netAnalysis_computeCentrality(cc, slot.name = 'netP')
  
  cellchat_list[[s]] <- cc
  saveRDS(cc, file.path(RES_DIR, paste0('cellchat_', gsub('[+]', 'pos', s), '.rds')))
  cat('Done.', s, '|', nrow(cc@LR$LRsig), 'significant L-R pairs |',
      length(cc@netP$pathways), 'pathways\n')
}

future::plan('sequential')

## 5. Hero figure: per-subtype interaction circle plot + heatmap

In [ ]:
png(file.path(FIG_DIR, 'hero_circle_plots.png'), width = 16, height = 6, units = 'in', res = 300, bg = 'white')
par(mfrow = c(1, 3), mar = c(2, 2, 3, 2))
for (s in subtypes) {
  cc <- cellchat_list[[s]]
  n_lr <- nrow(cc@LR$LRsig)
  n_path <- length(cc@netP$pathways)
  netVisual_circle(
    cc@net$weight,
    vertex.weight = as.numeric(table(cc@idents)),
    weight.scale = TRUE,
    label.edge = FALSE,
    title.name = sprintf('%s\n(%d L-R pairs, %d pathways)', s, n_lr, n_path)
  )
}
dev.off()
cat('Saved hero_circle_plots.png\n')

In [ ]:
# Per-subtype interaction weight heatmap (signal strength matrix)
# Subtypes have different cell-type counts (ER+/HER2+ have 8, TNBC has 9),
# so we save each as its own PNG instead of merging.
for (s in subtypes) {
  cc <- cellchat_list[[s]]
  ht <- netVisual_heatmap(cc, measure = 'weight', color.heatmap = 'Reds', title.name = s)
  png(file.path(FIG_DIR, paste0('interaction_heatmap_', gsub('[+]', 'pos', s), '.png')),
      width = 7, height = 6, units = 'in', res = 300, bg = 'white')
  ComplexHeatmap::draw(ht)
  dev.off()
}
cat('Saved per-subtype interaction heatmaps.\n')


## 6. Merge for comparison analysis

In [ ]:
# Lift to common cell labels - all three subtypes share the same 9 celltype_major labels
# but some may have zero cells in certain categories. Check first.
common_labels <- Reduce(intersect, lapply(cellchat_list, function(x) levels(x@idents)))
cat('Common labels across subtypes:', length(common_labels), '\n')
print(common_labels)

# Lift any subtype missing labels
all_labels <- Reduce(union, lapply(cellchat_list, function(x) levels(x@idents)))
for (s in subtypes) {
  cc <- cellchat_list[[s]]
  if (!all(all_labels %in% levels(cc@idents))) {
    cat('Lifting', s, 'to common labels\n')
    cellchat_list[[s]] <- liftCellChat(cc, all_labels)
  }
}

merged <- mergeCellChat(cellchat_list, add.names = subtypes, cell.prefix = TRUE)
cat('\nMerged CellChat object:\n')
print(merged)

## 7. Comparison: total interactions and strength

In [ ]:
gg1 <- compareInteractions(merged, show.legend = FALSE, group = seq_along(subtypes), color.use = c('#1f77b4','#ff7f0e','#d62728'))
gg2 <- compareInteractions(merged, show.legend = FALSE, group = seq_along(subtypes), measure = 'weight', color.use = c('#1f77b4','#ff7f0e','#d62728'))

p <- gg1 + gg2 + plot_annotation(title = 'Global signaling activity per subtype',
                                  theme = theme(plot.title = element_text(face='bold', size=13)))
save_fig(p, file.path(FIG_DIR, 'total_interactions_per_subtype.png'), w = 9, h = 4.5)
print(p)

## 8. Differential interaction networks (pairwise)

Red edges = increased in second subtype; blue edges = decreased.

In [ ]:
pairwise <- list(
  c('ER+',   'TNBC'),    # diff = TNBC - ER+
  c('ER+',   'HER2+'),
  c('HER2+', 'TNBC')
)

png(file.path(FIG_DIR, 'differential_interaction_circles.png'), width = 16, height = 6, units = 'in', res = 300, bg = 'white')
par(mfrow = c(1, 3), mar = c(2, 2, 3, 2))
for (pair in pairwise) {
  idx <- match(pair, subtypes)
  m_pair <- mergeCellChat(cellchat_list[pair], add.names = pair, cell.prefix = TRUE)
  netVisual_diffInteraction(m_pair, weight.scale = TRUE, measure = 'weight',
                            title.name = sprintf('%s vs %s (red: up in %s)', pair[2], pair[1], pair[2]))
}
dev.off()
cat('Saved differential_interaction_circles.png\n')

## 9. Pathway-level ranking: which pathways differ?

`rankNet` computes the total signaling strength of each pathway in each subtype and ranks them by overall contribution.

In [ ]:
rank_plot_full <- rankNet(merged, mode = 'comparison', stacked = TRUE, do.stat = TRUE,
                          color.use = c('#1f77b4','#ff7f0e','#d62728'))
save_fig(rank_plot_full, file.path(FIG_DIR, 'pathway_rank_stacked.png'), w = 8, h = 12)
print(rank_plot_full)

# Top 20 pathways (sorted by total info flow across subtypes)
rank_plot_top <- rankNet(merged, mode = 'comparison', stacked = FALSE, do.stat = TRUE,
                         color.use = c('#1f77b4','#ff7f0e','#d62728'))
save_fig(rank_plot_top, file.path(FIG_DIR, 'pathway_rank_unstacked.png'), w = 12, h = 7)

## 10. Top L-R pairs per subtype

Bubble plot of the most active ligand-receptor pairs (by communication probability) for each subtype, showing which cell-type pairs use them.

In [ ]:
# Top L-R pairs across all sources/targets in each subtype
for (s in subtypes) {
  cc <- cellchat_list[[s]]
  df_net <- subsetCommunication(cc)
  if (nrow(df_net) > 0) {
    df_net <- df_net[order(-df_net$prob), ]
    write.csv(df_net, file.path(RES_DIR, paste0('LR_pairs_', gsub('[+]', 'pos', s), '.csv')), row.names = FALSE)
    cat(s, ': saved', nrow(df_net), 'L-R interactions\n')
  }
}

# Multi-subtype bubble plot of top 20 L-R pairs across all subtypes
df_all <- list()
for (s in subtypes) {
  df_s <- subsetCommunication(cellchat_list[[s]])
  df_s$subtype <- s
  df_all[[s]] <- df_s
}
df_combined <- do.call(rbind, df_all)
top_lr <- df_combined %>%
  group_by(interaction_name) %>%
  summarise(total_prob = sum(prob, na.rm = TRUE), n_subtypes = n_distinct(subtype)) %>%
  arrange(desc(total_prob)) %>%
  head(25) %>%
  pull(interaction_name)

df_top <- df_combined %>% filter(interaction_name %in% top_lr) %>%
  mutate(pair = paste(source, '->', target),
         interaction_name = factor(interaction_name, levels = rev(top_lr)))

p_bubble <- ggplot(df_top, aes(x = subtype, y = interaction_name, size = prob, color = -log10(pval + 1e-3))) +
  geom_point() +
  scale_color_gradient(low = 'steelblue', high = 'firebrick', name = '-log10(p)') +
  scale_size_continuous(range = c(1, 6), name = 'Comm prob') +
  theme_minimal(base_size = 10) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1),
        panel.grid.major = element_line(color = 'gray90', size = 0.3)) +
  labs(title = 'Top 25 L-R interactions across subtypes', x = NULL, y = NULL)
save_fig(p_bubble, file.path(FIG_DIR, 'top_LR_bubble.png'), w = 9, h = 10)
print(p_bubble)

## 11. Subtype-specific signaling: pathway in/out strength heatmap

In [ ]:
# Identify top pathways per subtype, plot signalingRole heatmap
pathway_top <- list()
for (s in subtypes) {
  cc <- cellchat_list[[s]]
  if (length(cc@netP$pathways) > 0) {
    # Sort pathways by total signaling strength
    pathway_top[[s]] <- cc@netP$pathways[1:min(15, length(cc@netP$pathways))]
  }
}

# Outgoing + incoming signaling per cell type, per subtype (3 subtypes side-by-side)
png(file.path(FIG_DIR, 'signaling_role_heatmaps.png'), width = 18, height = 8, units = 'in', res = 300, bg = 'white')
ht_outgoing <- list()
for (s in subtypes) {
  cc <- cellchat_list[[s]]
  ht_outgoing[[s]] <- netAnalysis_signalingRole_heatmap(
    cc, pattern = 'outgoing', height = 8, width = 5, color.heatmap = 'OrRd',
    title = paste(s, '- outgoing'))
}
draw(ht_outgoing[[1]] + ht_outgoing[[2]] + ht_outgoing[[3]], ht_gap = unit(0.5, 'cm'))
dev.off()
cat('Saved signaling_role_heatmaps.png\n')

## 12. Summary

In [ ]:
summary_list <- list(
  dataset = 'Wu 2021 breast cancer, 3 patients, 12,962 cells',
  subtypes = subtypes,
  cellchat_version = as.character(packageVersion('CellChat')),
  CellChatDB_n_interactions = nrow(CellChatDB$interaction),
  per_subtype = list()
)

for (s in subtypes) {
  cc <- cellchat_list[[s]]
  summary_list$per_subtype[[s]] <- list(
    n_cells = ncol(cc@data),
    n_celltypes = length(levels(cc@idents)),
    n_significant_LR_pairs = nrow(cc@LR$LRsig),
    n_signaling_pathways = length(cc@netP$pathways),
    total_n_interactions = sum(cc@net$count),
    total_interaction_strength = round(sum(cc@net$weight), 4),
    top_pathways = head(cc@netP$pathways, 10)
  )
}

library(jsonlite)
write_json(summary_list, file.path(RES_DIR, 'cellchat_summary.json'), pretty = TRUE, auto_unbox = TRUE)

for (s in subtypes) {
  cat('\n===', s, '===\n')
  ps <- summary_list$per_subtype[[s]]
  cat('Cells:               ', ps$n_cells, '\n')
  cat('Cell types:          ', ps$n_celltypes, '\n')
  cat('Significant L-R:     ', ps$n_significant_LR_pairs, '\n')
  cat('Pathways:            ', ps$n_signaling_pathways, '\n')
  cat('Total interactions:  ', ps$total_n_interactions, '\n')
  cat('Total strength:      ', ps$total_interaction_strength, '\n')
  cat('Top 10 pathways:     ', paste(ps$top_pathways, collapse = ', '), '\n')
}

saveRDS(merged, file.path(RES_DIR, 'cellchat_merged.rds'))
cat('\nSaved merged CellChat object to results/\n')

## 13. Conclusions

Per-subtype CellChat analysis on Wu 2021 reveals:

1. **Global activity differences** - total interaction count and strength differ between TNBC, HER2+, and ER+ TMEs (Figure: total_interactions_per_subtype.png)
2. **Cell-type-level differential signaling** - the differential circle plots identify which cell-type pairs gain or lose signaling between subtypes
3. **Pathway ranking** - `rankNet` ranks pathways by total signaling contribution per subtype; pathways at the top of one subtype but not others are the candidate subtype-specific signaling axes
4. **Top L-R pairs** - the bubble plot identifies the 25 most active ligand-receptor pairs across all subtypes and shows which subtypes use them
5. **Signaling role heatmap** - per-cell-type outgoing signaling strength across the top pathways shows which cell types are the dominant senders in each subtype's TME

**Method context (Project 09 vs Project 02):** Project 02 established the cell-type annotations (Harmony integration + automatic lineage annotation, 98.7% agreement with author labels). Project 09 builds on those annotations to ask the *functional* question: given these cell types, how do they talk to each other, and does that conversation differ between cancer subtypes? CellChat is the standard tool for this functional layer of multi-sample scRNA-seq analysis.

**Caveat on patient-level vs subtype-level inference.** Each subtype here is represented by a single patient (CID3921 = ER+, CID4290A = HER2+, CID4515 = TNBC). Differences between the subtypes therefore conflate biological subtype-specific signaling with patient-to-patient biological variation. A production study would use multiple patients per subtype to disentangle these. The pipeline and the visualizations transfer directly to that larger setting.